# De bytes a FLOP/s: quando a GPU deixa de esperar pela memória?

Este notebook reproduz a ideia de **Reasoning about Memory-Bandwidth Costs**, de Horace He.

A GPU lê cada número de um vetor grande, multiplica esse número várias vezes e grava o resultado. A quantidade de dados é sempre a mesma; só aumentamos o número de multiplicações.

- **Poucas multiplicações:** o tempo quase não muda. A GPU passa a maior parte do tempo esperando a memória (limitada pela memória).
- **Muitas multiplicações:** o tempo começa a crescer. Agora o gargalo é o cálculo (limitada pelo cálculo).

O objetivo é encontrar onde acontece essa virada.


## Parâmetros do experimento

Ajuste estes valores antes de executar as células seguintes.

In [2]:
# 2**24 elementos = 64 MiB por tensor (em FP32);
# O tensor deve ser bem maior que o cache L2 da GPU. (Assim, reduzimos a influência do cache na medição da banda de memória.)
N = 2**24 # Tamanho do vetor/tensor
# Mede os valores iniciais e, a partir de 16, avança de 16 em 16 até 512.
REPEATS = [1, 2, 4, 8, *range(16, 513, 16)]
# Subconjunto usado nos três primeiros gráficos. O último usa toda a lista.
REPEATS_INICIAIS = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]

# Número de medições para cada valor de repeat (ou seja, quantidade de vezes que o experimento é feito).
N_SAMPLES = 100


In [3]:
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import triton
import triton.language as tl

if not torch.cuda.is_available():
    raise RuntimeError("Este experimento requer PyTorch com CUDA e uma GPU NVIDIA acessível.")

props = torch.cuda.get_device_properties(0)
metadata = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(), "torch": torch.__version__,
    "triton": triton.__version__, "cuda": torch.version.cuda,
    "gpu": props.name, "vram_bytes": props.total_memory,
}
print(json.dumps(metadata, indent=2))


{
  "timestamp_utc": "2026-11-06T15:17:58.072846+00:00",
  "python": "3.12.14",
  "torch": "2.15.0+cu132",
  "triton": "3.9.0",
  "cuda": "13.2",
  "gpu": "NVIDIA GeForce RTX 3050 Laptop GPU",
  "vram_bytes": 4007591936
}


In [4]:
ALPHA = 1.0001
BLOCK = 1024
WARMUP_RUNS = 5
SEED = 42

# Referências opcionais para linhas verdes nos gráficos. Use valores da SUA GPU.
# P deve corresponder a multiplicações FP32, não Tensor Cores ou TFLOP/s de FMA.
PEAK_BANDWIDTH_GBPS = None
PEAK_MUL_TFLOPS = None

OUT = Path("resultados_memory_bandwidth")
OUT.mkdir(exist_ok=True)
assert N > 0 and all(r > 0 for r in REPEATS) and N_SAMPLES > 0
assert 1.0 < ALPHA < 1.001  # Evita o overflow de multiplicar por 2 centenas de vezes.
free, _ = torch.cuda.mem_get_info()
assert 2 * N * 4 < free * 0.6, "Reduza N: precisamos de folga para CUDA e benchmark."
print(f"Tensor: {N:,} elementos; {N*4/2**20:.0f} MiB; intensidade: {REPEATS[0]/8:g} a {REPEATS[-1]/8:g} FLOP/byte")


Tensor: 16,777,216 elementos; 64 MiB; intensidade: 0.125 a 64 FLOP/byte


## 2. Garantir que o trabalho exista

No artigo, o exemplo usa `x = x * 2`. Repeti-lo 256 vezes em FP32 pode causar overflow. Além disso, um compilador pode simplificar expressões e invalidar nossa contagem de FLOP.

Usaremos `alpha = 1.0001` e uma instrução explícita `mul.rn.f32` em PTX dentro de Triton. Ela faz uma multiplicação FP32 com arredondamento a cada etapa. Isso mantém a cadeia de operações visível no código gerado. A parte avançada está apenas nesta célula: conceitualmente, o kernel **lê → multiplica r vezes em registradores → escreve**.

A cadeia tem dependências entre multiplicações; portanto, o teto observado pode refletir também latência e ocupação. Ele não é uma medição universal do pico de computação da GPU.


In [5]:
@triton.jit
def multiply_chain(X, Y, n_elements: tl.constexpr, alpha,
                   REPEAT: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    offsets = tl.program_id(0) * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    value = tl.load(X + offsets, mask=mask, other=0.0)
    for _ in tl.static_range(REPEAT):
        value = tl.inline_asm_elementwise(
            "mul.rn.f32 $0, $1, $2;",
            constraints="=f,f,f", args=[value, alpha],
            dtype=tl.float32, is_pure=True, pack=1,
        )
    tl.store(Y + offsets, value, mask=mask)


def fused(x, y, repeat):
    return multiply_chain[(triton.cdiv(x.numel(), BLOCK),)](
        x, y, x.numel(), ALPHA, REPEAT=repeat, BLOCK_SIZE=BLOCK,
    )



## 3. Validar valores e inspecionar a compilação

Antes de medir, verificamos os resultados numéricos para todos os valores de `repeat` usando uma referência calculada no NumPy, com arredondamento FP32 a cada multiplicação. Também contamos as instruções de multiplicação no PTX. O compilador pode processar vários elementos por thread, então a contagem estática pode ser um múltiplo de `repeat`.

Essa inspeção verifica o código intermediário PTX; uma investigação de instruções de máquina exigiria inspecionar também o SASS.


In [6]:
torch.manual_seed(SEED)
small_x = torch.rand(4099, device="cuda", dtype=torch.float32) * 0.5 + 0.5
small_y = torch.empty_like(small_x)
ptx_counts = {}
for r in REPEATS:
    reference = small_x.cpu().numpy().copy()
    for _ in range(r):
        np.multiply(reference, np.float32(ALPHA), out=reference)
    compiled = fused(small_x, small_y, r)
    torch.testing.assert_close(small_y.cpu(), torch.from_numpy(reference), rtol=1e-6, atol=1e-7)
    count = compiled.asm["ptx"].count("mul.rn.f32")
    assert count >= r and count % r == 0, (r, count)
    ptx_counts[r] = count
print("Resultados corretos. Instruções mul.rn.f32 no PTX:", ptx_counts)
(OUT / "kernel_repeat_max.ptx").write_text(compiled.asm["ptx"])
del small_x, small_y, reference


.venv/lib/python3.12/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


Resultados corretos. Instruções mul.rn.f32 no PTX: {1: 8, 2: 16, 4: 32, 8: 64, 16: 128, 32: 256, 48: 384, 64: 512, 80: 640, 96: 768, 112: 896, 128: 1024, 144: 1152, 160: 1280, 176: 1408, 192: 1536, 208: 1664, 224: 1792, 240: 1920, 256: 2048, 272: 2176, 288: 2304, 304: 2432, 320: 2560, 336: 2688, 352: 2816, 368: 2944, 384: 3072, 400: 3200, 416: 3328, 432: 3456, 448: 3584, 464: 3712, 480: 3840, 496: 3968, 512: 4096}


## 4. Protocolo de medição

A GPU trabalha de forma assíncrona. Cronometrar apenas a chamada Python pode medir o lançamento, sem esperar pelo resultado. Usamos eventos CUDA para medir exatamente `N_SAMPLES` execuções para cada valor de `repeat`. A compilação e as `WARMUP_RUNS` execuções de aquecimento acontecem **antes** dessas medições.

As medições não limpam o cache entre execuções. Usamos tensores grandes em relação ao cache L2 para reduzir sua influência. Os bytes calculados são o **tráfego lógico mínimo**, não contadores físicos de DRAM. Para medir os bytes físicos, seria necessário um profiler de hardware.

Primeiro medimos uma cópia como referência empírica de banda. Depois embaralhamos os valores de `repeat` para reduzir a associação entre intensidade e aquecimento da GPU. O intervalo P20–P80 descreve a dispersão das amostras; não é um intervalo de confiança.

Conecte o notebook à tomada e evite outras cargas na GPU. Os resultados de uma GPU de laptop dependem de temperatura, energia e clocks.


In [7]:
x = torch.rand(N, device="cuda", dtype=torch.float32) * 0.5 + 0.5
a = torch.empty_like(x)

# Compila e aquece todas as especializações fora da região cronometrada.
for r in REPEATS:
    fused(x, a, r)
torch.cuda.synchronize()


def measure(fn):
    # Aquecimento separado das medições.
    for _ in range(WARMUP_RUNS):
        fn()
    torch.cuda.synchronize()

    starts = [torch.cuda.Event(enable_timing=True) for _ in range(N_SAMPLES)]
    ends = [torch.cuda.Event(enable_timing=True) for _ in range(N_SAMPLES)]
    for start, end in zip(starts, ends):
        start.record()
        fn()
        end.record()
    torch.cuda.synchronize()

    samples = [start.elapsed_time(end) for start, end in zip(starts, ends)]
    median, p20, p80 = np.quantile(samples, [0.5, 0.2, 0.8])
    return float(median), float(p20), float(p80), len(samples)


copy_ms, _, _, _ = measure(lambda: a.copy_(x))
copy_gbps = 8 * N / (copy_ms * 1e-3) / 1e9
print(f"Cópia: {copy_ms*1000:.1f} µs; {copy_gbps:.1f} GB/s (referência empírica, não pico teórico)")

rows = []
rng = np.random.default_rng(SEED)
for r_raw in rng.permutation(REPEATS):
    r = int(r_raw)
    ms, p20, p80, sample_count = measure(lambda: fused(x, a, r))
    seconds = ms * 1e-3
    traffic = 8 * N
    rows.append({
        "repeat": r, "intensity_flop_byte": N*r/traffic,
        "median_ms": ms, "p20_ms": p20, "p80_ms": p80, "samples": sample_count,
        "logical_bytes": traffic, "flops": N*r,
        "tflops": N*r/seconds/1e12, "bandwidth_gbps": traffic/seconds/1e9,
    })
    print(f"r={r:3d}: {ms*1000:9.1f} µs | {rows[-1]['tflops']:.3f} TFLOP/s | {rows[-1]['bandwidth_gbps']:.1f} GB/s | {sample_count} amostras")
rows.sort(key=lambda row: row["repeat"])


Cópia: 735.2 µs; 182.6 GB/s (referência empírica, não pico teórico)
r=384:    2025.0 µs | 3.182 TFLOP/s | 66.3 GB/s | 100 amostras
r=240:    1197.6 µs | 3.362 TFLOP/s | 112.1 GB/s | 100 amostras
r=368:    1999.9 µs | 3.087 TFLOP/s | 67.1 GB/s | 100 amostras
r= 32:     736.3 µs | 0.729 TFLOP/s | 182.3 GB/s | 100 amostras
r= 16:     735.9 µs | 0.365 TFLOP/s | 182.4 GB/s | 100 amostras
r=320:    1754.9 µs | 3.059 TFLOP/s | 76.5 GB/s | 100 amostras
r=512:    2651.1 µs | 3.240 TFLOP/s | 50.6 GB/s | 100 amostras
r= 64:     736.3 µs | 1.458 TFLOP/s | 182.3 GB/s | 100 amostras
r=352:    1910.8 µs | 3.091 TFLOP/s | 70.2 GB/s | 100 amostras
r=464:    2437.3 µs | 3.194 TFLOP/s | 55.1 GB/s | 100 amostras
r=112:     742.5 µs | 2.531 TFLOP/s | 180.8 GB/s | 100 amostras
r=304:    1675.6 µs | 3.044 TFLOP/s | 80.1 GB/s | 100 amostras
r=272:    1368.1 µs | 3.336 TFLOP/s | 98.1 GB/s | 100 amostras
r=480:    2524.2 µs | 3.190 TFLOP/s | 53.2 GB/s | 100 amostras
r=208:    1047.6 µs | 3.331 TFLOP/s | 128.1 G

## 5. Os três gráficos do artigo

Os gráficos mostram o kernel fundido apenas para os valores de `REPEATS_INICIAIS`: 1, 2, 4, 8, 16, 32, 64, 128, 256 e 512. Nele, `repeat` é proporcional à intensidade aritmética, como no artigo. Ambos os eixos são logarítmicos.

A linha de cópia é uma referência empírica. Linhas de pico teórico aparecem apenas se você fornecer valores na configuração; não reutilizamos os números da A100 para outra GPU. Banda usa **GB/s**, FLOP/s usa **TFLOP/s**, e tempo usa **µs**.

Passe o mouse sobre um ponto para ler `repeat` e o valor daquele gráfico: tempo, computação ou banda. Você também pode ampliar regiões do gráfico.

Esta célula reutiliza `rows`, calculado na medição anterior. Requer Plotly (`.venv/bin/python -m pip install plotly==6.9.0`). O HTML interativo é salvo em `resultados_memory_bandwidth/tres_graficos_interativos.html` e pode ser aberto no navegador sem internet.


In [10]:
from math import log10
import plotly.graph_objects as go
from plotly.subplots import make_subplots

interactive_fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Tempo de execução", "Computação alcançada", "Banda lógica alcançada"],
    horizontal_spacing=0.08,
)
# Usa os resultados já medidos; esta célula não executa o benchmark novamente.
plot_rows = sorted(
    (row for row in rows if row["repeat"] in REPEATS_INICIAIS),
    key=lambda row: row["repeat"],
)
repeats = [row["repeat"] for row in plot_rows]
customdata = [
    [row["intensity_flop_byte"], row["median_ms"] * 1000,
     row["tflops"], row["bandwidth_gbps"], row["p20_ms"] * 1000,
     row["p80_ms"] * 1000, row["samples"]]
    for row in plot_rows
]
# Faixa de dispersão no gráfico de tempo.
for key, fill in [("p20_ms", None), ("p80_ms", "tonexty")]:
    interactive_fig.add_trace(go.Scatter(
        x=repeats, y=[row[key] * 1000 for row in plot_rows],
        mode="lines", line=dict(width=0), fill=fill,
        fillcolor="rgba(31,119,180,0.2)", name="P20–P80",
        showlegend=fill is not None, hoverinfo="skip",
    ), row=1, col=1)

for col, index, ylabel, tooltip in [
    (1, 1, "Tempo (µs)", "Tempo: %{y:.2f} µs"),
    (2, 2, "TFLOP/s", "Computação: %{y:.3f} TFLOP/s"),
    (3, 3, "GB/s", "Banda lógica: %{y:.2f} GB/s"),
]:
    interactive_fig.add_trace(go.Scatter(
        x=repeats, y=[values[index] for values in customdata],
        mode="lines+markers", name="Fundido", legendgroup="fundido",
        showlegend=col == 1, line=dict(color="#1f77b4"),
        marker=dict(size=8),
        hovertemplate="repeat = %{x}<br>" + tooltip + "<extra></extra>",
    ), row=1, col=col)
    interactive_fig.update_xaxes(
        type="log", title_text="repeat", tickmode="array",
        tickvals=repeats, ticktext=[str(r) for r in repeats],
        tickangle=-45, row=1, col=col,
    )
    interactive_fig.update_yaxes(type="log", title_text=ylabel, row=1, col=col)

interactive_fig.add_hline(
    y=copy_gbps, line_dash="dash", line_color="gray",
    annotation_text=f"Cópia medida: {copy_gbps:.1f} GB/s",
    # As anotações usam log10(y) em eixos logarítmicos; a linha usa y.
    annotation_y=log10(copy_gbps), row=1, col=3,
)
for col, peak in [(2, PEAK_MUL_TFLOPS), (3, PEAK_BANDWIDTH_GBPS)]:
    if peak is not None:
        interactive_fig.add_hline(
            y=peak, line_color="green", annotation_text="Pico informado",
            annotation_y=log10(peak), row=1, col=col,
        )
interactive_fig.update_layout(
    title=f"Intensidade crescente, tráfego de dados fixo — {props.name}",
    template="plotly_white", height=500, hovermode="closest",
    hoverlabel=dict(align="left"), margin=dict(t=100, b=100),
    legend=dict(orientation="h", y=-0.25),
)
# HTML independente: pode ser aberto no navegador e funciona sem internet.
interactive_fig.write_html(OUT / "tres_graficos_interativos.html", include_plotlyjs=True)
interactive_fig.show(renderer="plotly_mimetype")


## 6. Performance por intensidade aritmética

Cada elemento FP32 é lido e escrito uma vez: são **8 bytes** de tráfego lógico e `repeat` multiplicações. Assim, a intensidade aritmética é `repeat / 8` operações/byte. A performance é o número total de multiplicações (`N × repeat`) dividido pelo tempo mediano em segundos.

O gráfico usa todas as medições de `REPEATS` (1, 2, 4, 8 e de 16 até 512 em passos de 16), com ambos os eixos em escala linear. O HTML interativo é salvo em `resultados_memory_bandwidth/performance_por_intensidade.html`.

As duas linhas verticais tracejadas marcam o **ponto de virada teórico** $I^* = P/B$, com banda $B = 192$ GB/s (128 bits × 12 Gbps) e pico de multiplicações $P = 2048$ núcleos × clock. A linha da esquerda usa o *boost* nominal (1500 MHz, $I^* = 16{,}0$ op/byte); a da direita, o clock máximo do hardware (2100 MHz, $I^* = 22{,}4$ op/byte). As linhas horizontais tracejadas marcam os tetos de computação correspondentes, $P \approx 3{,}07$ e $4{,}30$ TFLOP/s. A faixa amarela entre os dois pontos de virada é a região de transição, que depende do clock que a GPU sustenta.

Também imprimimos, para comparação, uma estimativa feita a partir dos dados. Nela, testamos cada intensidade medida que tenha pelo menos três pontos de cada lado (incluindo a junção), ajustando duas retas conectadas por mínimos quadrados na escala linear. Buscamos uma inclinação inicial positiva seguida de menor inclinação, platô ou queda, e escolhemos a junção com o menor erro quadrático total. O critério BIC compara esse ajuste (quatro parâmetros, incluindo a posição da junção) com uma reta única (dois parâmetros); só imprimimos a estimativa se o ajuste segmentado tiver BIC menor. Sem evidência suficiente ou com menos de cinco pontos, nada é impresso.

Essa detecção descreve a mudança de inclinação da curva medida. Ela não comprova, isoladamente, uma transição física entre limitação por memória e por computação; ruído, latência e ocupação também influenciam o resultado. A resolução da estimativa depende do espaçamento entre as intensidades medidas.


In [11]:
# Reutiliza todas as medições de REPEATS, sem executar o benchmark novamente.
# Ponto de virada teórico: I* = P / B, com P = núcleos × clock
# (1 multiplicação FP32 por núcleo CUDA por ciclo). Valores da RTX 3050 Laptop.
CUDA_CORES = 2048
THEORETICAL_BANDWIDTH_GBPS = 192  # 128 bits × 12 Gbps (GDDR6)
TURNING_CLOCKS_MHZ = {"boost nominal": 1500, "clock máximo": 2100}
TURNING_COLORS = {"boost nominal": "#d62728", "clock máximo": "#1f4e9c"}
theoretical_turning_points = {
    name: CUDA_CORES * mhz * 1e6 / (THEORETICAL_BANDWIDTH_GBPS * 1e9)
    for name, mhz in TURNING_CLOCKS_MHZ.items()
}

roofline_rows = sorted(rows, key=lambda row: row["flops"] / row["logical_bytes"])
intensities = [row["flops"] / row["logical_bytes"] for row in roofline_rows]
performances = [row["flops"] / (row["median_ms"] * 1e-3) for row in roofline_rows]

def detect_turning_point(intensities, performances):
    """Estima uma mudança de inclinação por regressão linear segmentada contínua."""
    x = np.asarray(intensities, dtype=float)
    y = np.asarray(performances, dtype=float)
    if len(x) < 5:
        return None  # Exige pelo menos três pontos de cada lado, contando a junção.
    if (not np.all(np.isfinite(x)) or not np.all(np.isfinite(y))
            or np.any(np.diff(x) <= 0) or np.max(y) <= 0):
        raise ValueError("Use intensidades únicas crescentes e performances finitas positivas.")

    # Normaliza as unidades para estabilizar o ajuste numérico.
    x_scale, y_scale = x[-1] - x[0], np.max(y)
    u, v = (x - x[0]) / x_scale, y / y_scale
    baseline = np.column_stack([np.ones_like(u), u])
    baseline_fit = np.linalg.lstsq(baseline, v, rcond=None)[0]
    baseline_error = np.sum((v - baseline @ baseline_fit) ** 2)
    def bic(error, parameters):
        return len(x) * np.log(max(error / len(x), np.finfo(float).eps)) + parameters * np.log(len(x))

    best = None
    for i in range(2, len(x) - 2):
        # A função max(0, u - u[i]) mantém as duas retas conectadas na junção.
        design = np.column_stack([np.ones_like(u), u, np.maximum(0, u - u[i])])
        coefficients = np.linalg.lstsq(design, v, rcond=None)[0]
        slope_before = coefficients[1]
        slope_after = coefficients[1] + coefficients[2]
        if not slope_before > max(slope_after, 0):
            continue  # Busca crescimento seguido por menor ganho, platô ou queda.
        error = np.sum((v - design @ coefficients) ** 2)
        if best is None or error < best["error"]:
            best = {"index": i, "intensity": float(x[i]), "error": float(error)}

    # Quatro parâmetros: intercepto, duas inclinações e posição da junção.
    # Só marca se o BIC favorecer as duas retas sobre uma reta única (2 parâmetros).
    if best is None or bic(best["error"], 4) >= bic(baseline_error, 2):
        return None
    return best


turning_point = detect_turning_point(intensities, performances)
if turning_point is not None:
    print(f"Ponto de virada detectado pelo ajuste: {turning_point['intensity']:g} operações/byte.")

def region_points(low, high):
    """Pontos da curva entre low e high, com as bordas interpoladas."""
    inside = [(x, y) for x, y in zip(intensities, performances) if low < x < high]
    edges = [x for x in (low, high) if intensities[0] <= x <= intensities[-1]]
    edge_points = [(x, float(np.interp(x, intensities, performances))) for x in edges]
    return sorted(inside + edge_points)

low_turning, high_turning = sorted(theoretical_turning_points.values())
roofline_fig = go.Figure()
# As três áreas se encontram nas linhas tracejadas, sem lacunas.
# Rótulos: memory-bound encostado no canto inferior direito da sua região, onde a
# área sob a curva crescente é mais alta; compute-bound no centro da região.
for low, high, label, fillcolor, textcolor, label_place in [
    (-np.inf, low_turning, "memory-bound", "rgba(220, 53, 69, 0.28)", "#9b1c2c", "inferior direito"),
    (low_turning, high_turning, None, "rgba(255, 193, 7, 0.25)", None, None),
    (high_turning, np.inf, "compute-bound", "rgba(144, 238, 144, 0.50)", "#236b2b", "centro"),
]:
    points = region_points(low, high)
    if len(points) < 2:
        continue
    region_x = [x for x, _ in points]
    region_y = [y for _, y in points]
    roofline_fig.add_trace(go.Scatter(
        x=region_x, y=region_y, mode="lines", line=dict(width=0),
        fill="tozeroy", fillcolor=fillcolor,
        showlegend=False, hoverinfo="skip",
    ))
    if label is not None:
        span, top = region_x[-1] - region_x[0], max(region_y)
        if label_place == "inferior direito":
            position = dict(x=region_x[-1] - 0.03 * span, y=0.04 * top,
                            xanchor="right", yanchor="bottom")
        else:
            position = dict(x=region_x[0] + span / 2, y=0.45 * top)
        roofline_fig.add_annotation(
            **position, text=label, showarrow=False,
            font=dict(size=16, color=textcolor),
        )

roofline_fig.add_trace(go.Scatter(
    x=intensities, y=performances, mode="lines+markers",
    name="Performance medida", customdata=[row["repeat"] for row in roofline_rows],
    hovertemplate="Intensidade: %{x:.3g} operações/byte<br>"
                  "Performance: %{y:.3e} operações/s<br>"
                  "repeat: %{customdata}<extra></extra>",
))
# Rótulo do menor valor à esquerda da linha e do maior à direita, para não se sobreporem.
for (name, intensity), position in zip(
    sorted(theoretical_turning_points.items(), key=lambda item: item[1]),
    ["top left", "top right"],
):
    label = f"{intensity:.1f}".replace(".", ",")
    roofline_fig.add_vline(
        x=intensity, line_dash="dash", line_color=TURNING_COLORS[name],
        annotation_text=f"{TURNING_CLOCKS_MHZ[name]} MHz: {label} op/byte",
        annotation_font_color=TURNING_COLORS[name],
        annotation_position=position,
    )
    print(f"Ponto de virada teórico, {name} ({TURNING_CLOCKS_MHZ[name]} MHz): "
          f"{label} operações/byte (repeat ≈ {intensity * 8:.0f}).")

# Tetos de computação correspondentes: P = núcleos × clock.
for name, mhz in TURNING_CLOCKS_MHZ.items():
    peak = CUDA_CORES * mhz * 1e6
    roofline_fig.add_hline(
        y=peak, line_dash="dash", line_color=TURNING_COLORS[name],
        annotation_text=f"{mhz} MHz: {peak / 1e12:.2f} TFLOP/s".replace(".", ","),
        annotation_font_color=TURNING_COLORS[name],
        annotation_position="top right",
    )

roofline_fig.update_layout(
    title=f"Performance por intensidade aritmética — {props.name}",
    xaxis=dict(title="Intensidade aritmética (operações/byte)", type="linear"),
    yaxis=dict(title="Performance (operações/s)", type="linear"),
    template="plotly_white", hovermode="closest",
)
roofline_fig.write_html(OUT / "performance_por_intensidade.html", include_plotlyjs=True)
roofline_fig.show(renderer="plotly_mimetype")

Ponto de virada detectado pelo ajuste: 18 operações/byte.
Ponto de virada teórico, boost nominal (1500 MHz): 16,0 operações/byte (repeat ≈ 128).
Ponto de virada teórico, clock máximo (2100 MHz): 22,4 operações/byte (repeat ≈ 179).
